# USAERO – Phân tích dữ liệu BTS bằng Spark (EDA + KPI + bằng chứng kỹ thuật)

**Cách dùng:** chạy lần lượt từ trên xuống. `MONTHS` mặc định là 3 tháng đầu để chạy thử; đổi thành `FULL_MONTHS` ở cell cấu hình khi đã chắc đúng.

**Thiết kế:** mỗi tháng được tính **một lần** thành bảng cơ sở (các cột cộng được) → cuộn lên mọi mức ngày/tháng/năm (trong kỳ và xuyên kỳ) → tỷ lệ luôn tính lại từ tổng. Thay hoặc thêm dữ liệu một tháng chỉ cần chạy lại `build_months([(năm, tháng)])` rồi `build_kpis()`.

### 1. Môi trường

In [ ]:
# ── Môi trường: dùng pyspark có sẵn trong container nếu kernel chưa có ──
import os, sys, glob, re, time, json, io, contextlib, datetime as dt
if os.path.isdir("/opt/spark"):
    os.environ.setdefault("SPARK_HOME", "/opt/spark")
try:
    import pyspark
except ModuleNotFoundError:
    sys.path.insert(0, "/opt/spark/python")
    sys.path.extend(glob.glob("/opt/spark/python/lib/py4j-*-src.zip"))
    import pyspark
print("pyspark", pyspark.__version__, "| python", sys.version.split()[0])


pyspark 3.5.9 | python 3.8.10


### 2. Cấu hình, quy ước chỉ số và Spark session

In [13]:
from pyspark.sql import SparkSession, DataFrame, Window, functions as F, types as T

# ── Đường dẫn (có thể ghi đè bằng biến môi trường) ──
PROCESSED    = os.environ.get("AERO_PROCESSED", "/aero/processed")        # đầu ra của 06_batch_process_all.py
ANALYTICS    = os.environ.get("AERO_ANALYTICS", "/aero/analytics/eda")    # nơi ghi kết quả phân tích
HDFS_URI     = os.environ.get("AERO_HDFS", "hdfs://namenode:9000")
WAREHOUSE    = os.environ.get("AERO_WAREHOUSE", (HDFS_URI + "/aero/warehouse") if HDFS_URI else "/tmp/aero_warehouse")
EVIDENCE_DIR = os.environ.get("AERO_EVIDENCE", "./evidence")

def month_range(start, end):
    (y, m), (ey, em), out = start, end, []
    while (y, m) <= (ey, em):
        out.append((y, m)); m += 1
        if m > 12: y, m = y + 1, 1
    return out

DEV_MONTHS  = month_range((2022, 1), (2022, 3))     # chạy thử 3 tháng
FULL_MONTHS = month_range((2022, 1), (2026, 3))     # toàn bộ 51 tháng
MONTHS = FULL_MONTHS                                 # ← đổi thành FULL_MONTHS khi đã chạy đúng
_env = os.environ.get("AERO_MONTHS")                # ví dụ "2022-01:2023-03" hoặc "all"
if _env:
    _ym = lambda s: tuple(int(x) for x in s.split("-"))
    if _env == "all": MONTHS = FULL_MONTHS
    elif "," in _env: MONTHS = [_ym(s) for s in _env.split(",")]
    else: MONTHS = month_range(*[_ym(s) for s in _env.split(":")])

# ── Quy ước chỉ số (chỉ định nghĩa ở MỘT chỗ) ──
MIN_FLIGHTS = 200            # nhóm ít hơn ngưỡng này bị loại khi xếp hạng
MIN_BOX_N   = 30             # box-plot của nhóm nhỏ hơn ngưỡng này được gắn reliable=False
LATE_MIN    = 15             # chuyến "trễ" khi ARR_DELAY >= 15 phút (chuẩn DOT/BTS)
W_DELAY, W_SEVERE, W_CANCEL, W_DIVERT = 0.50, 0.20, 0.25, 0.05   # trọng số reliability_score (chỉ số tự định nghĩa)

def get_spark(app="usaero-eda", cores=2, mem="2g"):
    b = (SparkSession.builder.master(f"local[{cores}]").appName(app)
         .config("spark.driver.memory", mem)
         .config("spark.sql.shuffle.partitions", os.environ.get("AERO_SHUFFLE", "16"))
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .config("spark.sql.warehouse.dir", WAREHOUSE))
    if HDFS_URI:
        b = b.config("spark.hadoop.fs.defaultFS", HDFS_URI)
    s = b.getOrCreate()
    s.sparkContext.setLogLevel("WARN")
    return s

spark = get_spark()
print("Spark", spark.version, "| tháng cần xử lý:", f"{MONTHS[0][0]}-{MONTHS[0][1]:02d} → {MONTHS[-1][0]}-{MONTHS[-1][1]:02d}", f"({len(MONTHS)} tháng)")


Spark 3.5.3 | tháng cần xử lý: 2022-01 → 2026-03 (51 tháng)


### 3. Nạp dữ liệu theo tháng (partition pruning)

In [14]:
# ── Nạp dữ liệu theo tháng (partition pruning: chỉ đọc đúng thư mục year=/month= cần thiết) ──
def _hpath(p):
    path = spark._jvm.org.apache.hadoop.fs.Path(p)
    return path, path.getFileSystem(spark._jsc.hadoopConfiguration())

def month_dir(y, m):
    for name in (f"month={m:02d}", f"month={m}"):
        p = f"{PROCESSED}/year={y}/{name}"
        path, fs = _hpath(p)
        if fs.exists(path):
            return p
    return None

def load_months(months, verbose=True):
    found, missing = [], []
    for y, m in months:
        p = month_dir(y, m)
        if p: found.append(p)
        else: missing.append(f"{y}-{m:02d}")
    if missing and verbose:
        print(f"⚠ Thiếu {len(missing)} tháng (bỏ qua): {', '.join(missing)}")
    if not found:
        raise FileNotFoundError(f"Không tìm thấy partition nào dưới {PROCESSED}")
    return spark.read.option("basePath", PROCESSED).parquet(*found)

def peek(df, n=15):
    try:
        pdf = df.limit(n).toPandas()
        try: display(pdf)
        except NameError: print(pdf.to_string())
    except Exception:
        df.show(n, truncate=False)

def timed(fn):
    t = time.time(); r = fn(); return r, round(time.time() - t, 2)


### 4. Chuẩn hóa dữ liệu (một lần, một chỗ)

- `ARR_DELAY`/`DEP_DELAY` null = chuyến hủy hoặc chuyển hướng → **không** điền 0.
- Cột nguyên nhân trễ null = chuyến không trễ ≥15 phút → điền 0.

In [15]:
# ── Bảng tra nhỏ (dùng cho broadcast join) ──
CARRIERS = {"AA": "American", "DL": "Delta", "UA": "United", "WN": "Southwest", "B6": "JetBlue", "AS": "Alaska",
            "NK": "Spirit", "F9": "Frontier", "G4": "Allegiant", "HA": "Hawaiian", "YX": "Republic", "OO": "SkyWest",
            "MQ": "Envoy", "9E": "Endeavor", "OH": "PSA", "YV": "Mesa", "QX": "Horizon", "ZW": "Air Wisconsin",
            "C5": "CommutAir", "G7": "GoJet", "CP": "Compass", "SY": "Sun Country", "MX": "Breeze", "9K": "Cape Air"}

def carrier_lookup():
    return spark.createDataFrame(list(CARRIERS.items()), ["carrier", "carrier_name"])

def _nth_weekday(y, m, weekday, n):             # weekday: 0=Thứ hai; n>0: thứ n; n=-1: thứ cuối cùng
    if n > 0:
        d = dt.date(y, m, 1)
        return d + dt.timedelta(days=(weekday - d.weekday()) % 7 + 7 * (n - 1))
    d = dt.date(y + (m == 12), (m % 12) + 1, 1) - dt.timedelta(days=1)
    return d - dt.timedelta(days=(d.weekday() - weekday) % 7)

def holiday_df(years=range(2021, 2028), window=3):
    best = {}
    for y in years:
        hol = {"new_year": dt.date(y, 1, 1), "memorial": _nth_weekday(y, 5, 0, -1), "july4": dt.date(y, 7, 4),
               "labor": _nth_weekday(y, 9, 0, 1), "thanksgiving": _nth_weekday(y, 11, 3, 4), "christmas": dt.date(y, 12, 25)}
        for name, d0 in hol.items():
            for off in range(-window, window + 1):
                d = d0 + dt.timedelta(days=off)
                if d not in best or abs(off) < abs(best[d][1]):
                    best[d] = (name, off)
    schema = T.StructType([T.StructField("fl_date", T.DateType()), T.StructField("holiday", T.StringType()),
                           T.StructField("holiday_offset", T.IntegerType())])
    return spark.createDataFrame([(d, n, o) for d, (n, o) in best.items()], schema)

def severity_expr():
    a = F.col("arr_delay")
    return (F.when(F.col("cancelled") == 1, "1_cancelled").when(F.col("diverted") == 1, "2_diverted")
             .when(a.isNull(), "0_unknown").when(a < 0, "3_early").when(a < 15, "4_ontime")
             .when(a < 30, "5_minor").when(a < 60, "6_moderate").when(a < 180, "7_major").otherwise("8_severe"))

# ── Tên cột có thể khác nhau giữa các lần tải: khai báo danh sách tên ứng viên cho mỗi trường logic ──
COLS = {
    "fl_date": ["FL_DATE", "FlightDate"], "carrier": ["OP_UNIQUE_CARRIER", "OP_CARRIER", "REPORTING_AIRLINE", "UNIQUE_CARRIER", "CARRIER"],
    "origin": ["ORIGIN"], "dest": ["DEST"], "crs_dep_time": ["CRS_DEP_TIME"], "dep_delay": ["DEP_DELAY"], "arr_delay": ["ARR_DELAY"],
    "cancelled": ["CANCELLED"], "diverted": ["DIVERTED"], "cancel_code": ["CANCELLATION_CODE"],
    "taxi_out": ["TAXI_OUT"], "taxi_in": ["TAXI_IN"], "crs_elapsed": ["CRS_ELAPSED_TIME"], "actual_elapsed": ["ACTUAL_ELAPSED_TIME"],
    "distance": ["DISTANCE"], "tail": ["TAIL_NUM"],
    "d_carrier": ["CARRIER_DELAY"], "d_weather": ["WEATHER_DELAY"], "d_nas": ["NAS_DELAY"],
    "d_security": ["SECURITY_DELAY"], "d_late_aircraft": ["LATE_AIRCRAFT_DELAY"],
}
REQUIRED = ["fl_date", "carrier", "origin", "dest", "dep_delay", "arr_delay", "cancelled"]
CAUSES = ["carrier", "weather", "nas", "security", "late_aircraft"]
COLMAP = {}

def prepare(raw, verbose=True):
    """Chuẩn hóa MỘT lần: đổi tên cột, ép kiểu, xử lý null đúng ngữ nghĩa, tạo cột dẫn xuất."""
    global COLMAP
    low = {c.lower(): c for c in raw.columns}
    m = {k: next((low[c.lower()] for c in cands if c.lower() in low), None) for k, cands in COLS.items()}
    miss = [k for k in REQUIRED if m[k] is None]
    if miss:
        raise ValueError(f"Thiếu cột bắt buộc {miss}. Các cột hiện có: {raw.columns}")
    COLMAP = m
    if verbose:
        absent = [k for k, v in m.items() if v is None]
        if absent: print("ℹ Cột tùy chọn không có (chỉ số liên quan sẽ là null):", absent)

    def c(k, t="double"):
        return F.col(f"`{m[k]}`").cast(t) if m[k] else F.lit(None).cast(t)

    d = raw.select(
        F.to_date(F.col(f"`{m['fl_date']}`")).alias("fl_date"),
        F.col(f"`{m['carrier']}`").cast("string").alias("carrier"),
        F.col(f"`{m['origin']}`").cast("string").alias("origin"),
        F.col(f"`{m['dest']}`").cast("string").alias("dest"),
        c("tail", "string").alias("tail"),
        c("crs_dep_time", "int").alias("crs_dep_time"),
        c("dep_delay").alias("dep_delay"), c("arr_delay").alias("arr_delay"),
        F.coalesce(c("cancelled", "int"), F.lit(0)).alias("cancelled"),
        F.coalesce(c("diverted", "int"), F.lit(0)).alias("diverted"),
        c("cancel_code", "string").alias("cancel_code"),
        c("taxi_out").alias("taxi_out"), c("taxi_in").alias("taxi_in"),
        c("crs_elapsed").alias("crs_elapsed"), c("actual_elapsed").alias("actual_elapsed"), c("distance").alias("distance"),
        # Cột nguyên nhân: null nghĩa là chuyến không trễ ≥15' → điền 0 là đúng ngữ nghĩa
        *[F.coalesce(c(f"d_{k}"), F.lit(0.0)).alias(f"d_{k}") for k in CAUSES],
    ).where(F.col("fl_date").isNotNull())

    d = (d.withColumn("year", F.year("fl_date")).withColumn("month", F.month("fl_date")).withColumn("day", F.dayofmonth("fl_date"))
          .withColumn("dow", ((F.dayofweek("fl_date") + 5) % 7) + 1)                      # 1=Thứ hai … 7=Chủ nhật
          .withColumn("is_weekend", (F.col("dow") >= 6).cast("int"))
          .withColumn("dep_hour", (F.floor(F.col("crs_dep_time") / 100) % 24).cast("int"))
          .withColumn("dep_min", (F.col("crs_dep_time") % 100).cast("int"))
          .withColumn("dep_bucket", F.when(F.col("dep_hour") < 6, "night").when(F.col("dep_hour") < 12, "morning")
                                     .when(F.col("dep_hour") < 18, "afternoon").otherwise("evening"))
          .withColumn("dist_bucket", F.when(F.col("distance").isNull(), "unknown").when(F.col("distance") < 500, "short")
                                      .when(F.col("distance") < 1500, "medium").otherwise("long"))
          .withColumn("route", F.concat_ws("-", "origin", "dest"))
          .withColumn("operated", ((F.col("cancelled") == 0) & (F.col("diverted") == 0)).cast("int"))
          # delay chỉ có nghĩa khi chuyến thực sự bay: hủy → không có DEP/ARR delay, chuyển hướng → không có ARR delay
          # (KHÔNG điền 0, nếu không chuyến hủy sẽ bị tính là đúng giờ)
          .withColumn("arr_delay_ok", F.when(F.col("operated") == 1, F.col("arr_delay")))
          .withColumn("dep_delay_ok", F.when(F.col("cancelled") == 0, F.col("dep_delay")))
          .withColumn("air_gain", F.when(F.col("arr_delay_ok").isNotNull() & F.col("dep_delay_ok").isNotNull(),
                                         F.col("dep_delay_ok") - F.col("arr_delay_ok")))
          .withColumn("tier", severity_expr()))
    d = (d.join(F.broadcast(holiday_df()), "fl_date", "left")
          .withColumn("holiday", F.coalesce("holiday", F.lit("none")))
          .withColumn("holiday_offset", F.coalesce("holiday_offset", F.lit(0))))
    return d


### 5. Lõi KPI (cộng được → cuộn lên → tính tỷ lệ), UDF và custom aggregation

In [16]:
# ── Lõi KPI: tính các cột CỘNG ĐƯỢC (additive) một lần, sau đó cuộn lên (rollup) và tính tỷ lệ từ tổng ──
def _add_exprs():
    a, dp = F.col("arr_delay_ok"), F.col("dep_delay_ok")
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0))
    E = [("flights", F.count(F.lit(1))), ("cancelled", F.sum("cancelled")), ("diverted", F.sum("diverted")),
         ("operated", F.sum("operated")), ("arr_n", F.count(a)), ("on_time", cnt(a < LATE_MIN)), ("early_n", cnt(a < 0)),
         ("delayed15", cnt(a >= 15)), ("delayed30", cnt(a >= 30)), ("delayed60", cnt(a >= 60)), ("delayed180", cnt(a >= 180)),
         ("sum_arr_delay", F.sum(a)), ("sum_arr_delay_pos", F.sum(F.when(a > 0, a).otherwise(0.0))),
         ("sum_arr_delay_late", F.sum(F.when(a >= LATE_MIN, a).otherwise(0.0))), ("sum_early_min", F.sum(F.when(a < 0, -a).otherwise(0.0))),
         ("dep_n", F.count(dp)), ("dep_delayed15", cnt(dp >= 15)), ("sum_dep_delay", F.sum(dp)),
         ("sum_dep_delay_pos", F.sum(F.when(dp > 0, dp).otherwise(0.0)))]
    E += [(f"cause_{k}", F.sum(f"d_{k}")) for k in CAUSES]
    E += [("taxi_out_n", F.count("taxi_out")), ("sum_taxi_out", F.sum("taxi_out")), ("taxi_in_n", F.count("taxi_in")),
          ("sum_taxi_in", F.sum("taxi_in")), ("gain_n", F.count("air_gain")), ("sum_air_gain", F.sum("air_gain")),
          ("dist_n", F.count("distance")), ("sum_distance", F.sum("distance"))]
    E += [(f"cancel_{k}", cnt(F.col("cancel_code") == k.upper())) for k in "abcd"]     # A=hãng, B=thời tiết, C=NAS, D=an ninh
    return E

ADD_COLS = [n for n, _ in _add_exprs()]

def kpi_base(df, keys):
    return df.groupBy(*keys).agg(*[e.alias(n) for n, e in _add_exprs()])

def rollup(base, keys):
    n = F.sum("n_base_rows") if "n_base_rows" in base.columns else F.count(F.lit(1))
    return base.groupBy(*keys).agg(n.alias("n_base_rows"), *[F.sum(c).alias(c) for c in ADD_COLS])

def _r(num, den):
    return F.when(F.col(den) > 0, F.col(num) / F.col(den))

def finalize(df):
    """Tỷ lệ luôn tính lại từ tổng (không lấy trung bình của các tỷ lệ)."""
    rates = {
        "cancel_rate": _r("cancelled", "flights"), "divert_rate": _r("diverted", "flights"),
        "otp_operated": _r("on_time", "arr_n"),         # đúng giờ trên các chuyến đã đo được giờ đến
        "otp_scheduled": _r("on_time", "flights"),      # đúng giờ trên TẤT CẢ chuyến lên lịch (hủy/chuyển hướng = không đúng giờ)
        "early_rate": _r("early_n", "arr_n"), "delay15_rate": _r("delayed15", "arr_n"), "delay30_rate": _r("delayed30", "arr_n"),
        "severe60_rate": _r("delayed60", "arr_n"), "severe180_rate": _r("delayed180", "arr_n"),
        "avg_arr_delay_net": _r("sum_arr_delay", "arr_n"), "avg_arr_delay_pos": _r("sum_arr_delay_pos", "arr_n"),
        "avg_delay_when_late": _r("sum_arr_delay_late", "delayed15"), "avg_early_min": _r("sum_early_min", "early_n"),
        "dep_delay15_rate": _r("dep_delayed15", "dep_n"), "avg_dep_delay_net": _r("sum_dep_delay", "dep_n"),
        "avg_dep_delay_pos": _r("sum_dep_delay_pos", "dep_n"), "avg_taxi_out": _r("sum_taxi_out", "taxi_out_n"),
        "avg_taxi_in": _r("sum_taxi_in", "taxi_in_n"), "avg_air_gain": _r("sum_air_gain", "gain_n"),
        "avg_distance": _r("sum_distance", "dist_n"),
        "cancel_share_carrier": _r("cancel_a", "cancelled"), "cancel_share_weather": _r("cancel_b", "cancelled"),
        "cancel_share_nas": _r("cancel_c", "cancelled"), "cancel_share_security": _r("cancel_d", "cancelled"),
    }
    out = df.select("*", *[v.alias(k) for k, v in rates.items()])
    cc = [f"cause_{k}" for k in CAUSES]
    out = out.withColumn("cause_total", sum(F.coalesce(F.col(c), F.lit(0.0)) for c in cc))
    out = out.select("*", *[F.when(F.col("cause_total") > 0, F.col(c) / F.col("cause_total")).alias(c + "_share") for c in cc])
    z = lambda c: F.coalesce(F.col(c), F.lit(0.0))
    pen = W_DELAY * z("delay15_rate") + W_SEVERE * z("severe60_rate") + W_CANCEL * z("cancel_rate") + W_DIVERT * z("divert_rate")
    out = out.withColumn("reliability_score", F.greatest(F.lit(0.0), F.least(F.lit(100.0), 100 * (1 - pen))))
    return out.withColumn("is_reliable_sample", F.col("flights") >= MIN_FLIGHTS)

# ── UDF có lý do nghiệp vụ: phân tầng mức độ nghiêm trọng, đối chiếu với biểu thức tham chiếu ──
def _severity_py(a, cancelled, diverted):
    if cancelled == 1: return "1_cancelled"
    if diverted == 1: return "2_diverted"
    if a is None: return "0_unknown"
    if a < 0: return "3_early"
    if a < 15: return "4_ontime"
    if a < 30: return "5_minor"
    if a < 60: return "6_moderate"
    if a < 180: return "7_major"
    return "8_severe"
severity_udf = F.udf(_severity_py, T.StringType())

# ── Custom aggregation: trung bình cắt 1% hai đầu (bền vững với ngoại lệ) – pandas grouped-aggregate UDF ──
try:
    import pandas as pd
    @F.pandas_udf("double")
    def trimmed_mean(v: pd.Series) -> float:
        v = v.dropna()
        if len(v) == 0:
            return float("nan")
        lo, hi = v.quantile(0.01), v.quantile(0.99)
        return float(v[(v >= lo) & (v <= hi)].mean())
except Exception as _e:
    trimmed_mean = None
    print("ℹ Không dùng được pandas UDF (thiếu pandas/pyarrow):", _e)


### 6. Thống kê kiểu box-plot (chỉ tính số, không vẽ)

In [17]:
# ── Thống kê kiểu box-plot (chỉ tính số, không vẽ) ──
BOX_Q = [0.05, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]

def box_metric_exprs():
    a, d = F.col("arr_delay_ok"), F.col("dep_delay_ok")
    m = {"arr_delay_all": a, "arr_delay_late": F.when(a >= LATE_MIN, a),
         "dep_delay_all": d, "dep_delay_late": F.when(d >= LATE_MIN, d)}
    for k in CAUSES:                                   # phút trễ theo nguyên nhân, chỉ xét chuyến trễ ≥15'
        m[f"cause_{k}_late"] = F.when(a >= LATE_MIN, F.col(f"d_{k}"))
    return m

BOX_SETS = {"delay": ["arr_delay_all", "arr_delay_late", "dep_delay_all", "dep_delay_late"],
            "causes": [f"cause_{k}_late" for k in CAUSES]}

def boxplot_stats(df, keys, expr, label, min_n=MIN_BOX_N, accuracy=10000):
    """Lượt 1: Q1/median/Q3, hàng rào Tukey. Lượt 2: broadcast join hàng rào ngược vào dữ liệu để lấy đầu râu thật và đếm ngoại lệ."""
    base = df.select(*keys, expr.alias("v")).where(F.col("v").isNotNull())
    qs = ",".join(str(q) for q in BOX_Q)
    st = (base.groupBy(*keys).agg(
              F.count("v").alias("n"), F.min("v").alias("min"), F.max("v").alias("max"),
              F.mean("v").alias("mean"), F.stddev("v").alias("std"),
              F.expr(f"percentile_approx(v, array({qs}), {accuracy})").alias("q"))
          .select(*keys, "n", "min", "max", "mean", "std",
                  F.col("q")[0].alias("p05"), F.col("q")[1].alias("q1"), F.col("q")[2].alias("median"),
                  F.col("q")[3].alias("q3"), F.col("q")[4].alias("p90"), F.col("q")[5].alias("p95"), F.col("q")[6].alias("p99"))
          .withColumn("iqr", F.col("q3") - F.col("q1"))
          .withColumn("fence_low", F.col("q1") - 1.5 * F.col("iqr"))
          .withColumn("fence_high", F.col("q3") + 1.5 * F.col("iqr")))
    fences = st.select(*keys, "fence_low", "fence_high")
    j = base.join(F.broadcast(fences), keys, "inner") if keys else base.crossJoin(F.broadcast(fences))
    w = j.groupBy(*keys).agg(
        F.min(F.when(F.col("v") >= F.col("fence_low"), F.col("v"))).alias("whisker_low"),
        F.max(F.when(F.col("v") <= F.col("fence_high"), F.col("v"))).alias("whisker_high"),
        F.sum(F.when(F.col("v") < F.col("fence_low"), 1).otherwise(0)).alias("n_out_low"),
        F.sum(F.when(F.col("v") > F.col("fence_high"), 1).otherwise(0)).alias("n_out_high"))
    out = st.join(w, keys, "left") if keys else st.crossJoin(w)
    return (out.withColumn("out_low_rate", F.col("n_out_low") / F.col("n"))
               .withColumn("out_high_rate", F.col("n_out_high") / F.col("n"))
               .withColumn("metric", F.lit(label)).withColumn("reliable", F.col("n") >= min_n))

def boxplot_many(df, keys, set_name):
    exprs = box_metric_exprs()
    parts = [boxplot_stats(df, keys, exprs[lab], lab) for lab in BOX_SETS[set_name]]
    out = parts[0]
    for p in parts[1:]:
        out = out.unionByName(p)
    return out


### 7. Pipeline xây dựng bảng kết quả

In [18]:
# ── Pipeline: mỗi tháng tính MỘT lần → bảng cơ sở theo (year, month); thay/thêm dữ liệu chỉ cần chạy lại tháng đó ──
GRAINS = {"day": ["year", "month", "day"],              # chuỗi ngày: xem 1 tháng (hướng A) hoặc cả năm (hướng B-i)
          "calendar_day": ["month", "day"],             # cùng ngày lịch qua các năm (hướng B-ii)
          "month": ["year", "month"],                   # các tháng trong 1 năm (A) / chuỗi tháng (B)
          "month_of_year": ["month"],                   # cùng một tháng qua các năm → mùa vụ
          "year": ["year"]}
DIMS = {"overall": [], "carrier": ["carrier"], "origin": ["origin"]}
BOX_GRAINS = dict(GRAINS, dow=["dow"], hour=["dep_hour"])
BOX_DIMS = dict(DIMS, route=["origin", "dest"])

BASE_SPECS = {"day_overall": ["year", "month", "day"], "day_carrier": ["year", "month", "day", "carrier"],
              "day_origin": ["year", "month", "day", "origin"], "month_route": ["year", "month", "origin", "dest"],
              "month_carrier_origin": ["year", "month", "carrier", "origin"],
              "prof_hour": ["year", "month", "dep_hour"], "prof_hour_carrier": ["year", "month", "dep_hour", "carrier"],
              "prof_dow": ["year", "month", "dow"], "prof_dist": ["year", "month", "dist_bucket"],
              "prof_holiday": ["year", "month", "holiday", "holiday_offset"]}

# Box-plot: ngày/tháng chỉ cần dữ liệu của chính tháng đó → tính trong vòng lặp tháng; năm/mùa vụ cần nhiều tháng → tính sau
BOX_MONTH_PLAN = [("day", "overall", "delay"), ("day", "carrier", "delay"), ("month", "overall", "delay"),
                  ("month", "carrier", "delay"), ("month", "origin", "delay"), ("month", "overall", "causes")]
BOX_COARSE_PLAN = [("year", "overall", "delay"), ("year", "carrier", "delay"), ("year", "origin", "delay"), ("year", "route", "delay"),
                   ("year", "overall", "causes"), ("year", "carrier", "causes"),
                   ("month_of_year", "overall", "delay"), ("month_of_year", "carrier", "delay"),
                   ("calendar_day", "overall", "delay"), ("dow", "overall", "delay"), ("hour", "overall", "delay")]

def write_part(df, sub):
    df.coalesce(1).write.mode("overwrite").partitionBy("year", "month").parquet(f"{ANALYTICS}/{sub}")

def build_month(y, m, with_box=True):
    t0 = time.time()
    df = prepare(load_months([(y, m)], verbose=False), verbose=False).cache()
    n = df.count()
    for name, keys in BASE_SPECS.items():
        write_part(kpi_base(df, keys), f"base/{name}")
    write_part(df.groupBy("year", "month", "carrier", "tier").count(), "base/tier_carrier")
    if with_box:
        for grain, dim, mset in BOX_MONTH_PLAN:
            write_part(boxplot_many(df, BOX_GRAINS[grain] + BOX_DIMS[dim], mset), f"box/{grain}_{dim}_{mset}")
    df.unpersist()
    return n, round(time.time() - t0, 1)

def build_months(months, with_box=True):
    ok = [(y, m) for y, m in months if month_dir(y, m)]
    skipped = [f"{y}-{m:02d}" for y, m in months if not month_dir(y, m)]
    if skipped: print(f"⚠ Bỏ qua tháng không có dữ liệu: {', '.join(skipped)}")
    for y, m in ok:
        n, sec = build_month(y, m, with_box)
        print(f"  {y}-{m:02d}: {n:>10,} dòng | {sec:>6}s")
    return ok

def build_kpis():
    """Cuộn các bảng cơ sở lên mọi mức (ngày/tháng/năm × trong kỳ/xuyên kỳ) rồi tính tỷ lệ từ tổng."""
    def emit(name, base_name, keys):
        b = spark.read.parquet(f"{ANALYTICS}/base/{base_name}")
        finalize(rollup(b, keys)).coalesce(1).write.mode("overwrite").parquet(f"{ANALYTICS}/kpi/{name}")
    for dim, (bname, extra) in {"overall": ("day_overall", []), "carrier": ("day_carrier", ["carrier"]),
                                "origin": ("day_origin", ["origin"])}.items():
        for grain, gk in GRAINS.items():
            emit(f"{grain}_{dim}", bname, gk + extra)
    for grain in ("month", "year", "month_of_year"):
        emit(f"{grain}_route", "month_route", GRAINS[grain] + ["origin", "dest"])
    for grain in ("month", "year"):
        emit(f"{grain}_carrier_origin", "month_carrier_origin", GRAINS[grain] + ["carrier", "origin"])
    emit("hour_overall", "prof_hour", ["dep_hour"]); emit("hour_by_year", "prof_hour", ["year", "dep_hour"])
    emit("hour_carrier", "prof_hour_carrier", ["carrier", "dep_hour"])
    emit("dow_overall", "prof_dow", ["dow"]); emit("dow_by_year", "prof_dow", ["year", "dow"])
    emit("dist_overall", "prof_dist", ["dist_bucket"]); emit("holiday_overall", "prof_holiday", ["holiday", "holiday_offset"])

def build_box_coarse(months):
    df = prepare(load_months(months, verbose=False), verbose=False)
    for grain, dim, mset in BOX_COARSE_PLAN:
        t = time.time()
        out = boxplot_many(df, BOX_GRAINS[grain] + BOX_DIMS[dim], mset)
        out.coalesce(1).write.mode("overwrite").parquet(f"{ANALYTICS}/box/{grain}_{dim}_{mset}")
        print(f"  box/{grain}_{dim}_{mset}: {round(time.time() - t, 1)}s")

def kpi(name):  return spark.read.parquet(f"{ANALYTICS}/kpi/{name}")
def box(name):  return spark.read.parquet(f"{ANALYTICS}/box/{name}")
def base_tbl(name): return spark.read.parquet(f"{ANALYTICS}/base/{name}")


### 8. Các hàm phân tích và insight

In [19]:
# ═══════════ PHÂN TÍCH & INSIGHT (đọc từ các bảng KPI đã tính; mỗi hàm trả về DataFrame nhỏ) ═══════════

def overview():
    """Bức tranh tổng thể theo năm (năm cuối có thể chưa đủ 12 tháng: xem n_base_rows = số ngày có dữ liệu)."""
    return (kpi("year_overall").withColumn("flights_per_day", F.col("flights") / F.col("n_base_rows")).orderBy("year")
            .select("year", "n_base_rows", "flights", "flights_per_day", "cancel_rate", "divert_rate", "otp_operated", "otp_scheduled",
                    "avg_arr_delay_net", "avg_arr_delay_pos", "avg_delay_when_late", "delay15_rate", "severe60_rate",
                    "severe180_rate", "early_rate", "avg_air_gain", "reliability_score"))

# ── Mức THÁNG ──
def monthly_trend(metric="otp_scheduled"):
    """Chuỗi tháng + trung bình trượt 3 tháng (window) + so cùng kỳ năm trước (self-join theo t-12)."""
    cur = kpi("month_overall").withColumn("t", F.col("year") * 12 + F.col("month")).select("year", "month", "t", "flights", metric)
    prev = cur.select((F.col("t") + 12).alias("t"), F.col(metric).alias(f"{metric}_prev_year"))
    w = Window.orderBy("t").rangeBetween(-2, 0)
    return (cur.join(prev, "t", "left").withColumn(f"{metric}_ma3", F.avg(metric).over(w))
               .withColumn("yoy_change", F.col(metric) - F.col(f"{metric}_prev_year")).orderBy("t").drop("t"))

def seasonality(metric="delay15_rate"):
    """Chỉ số mùa vụ = giá trị của tháng-trong-năm / giá trị chung của toàn kỳ (>1: tháng xấu hơn bình thường nếu là chỉ số 'xấu')."""
    glob_ = finalize(rollup(kpi("year_overall"), [])).select(F.col(metric).alias("overall_value"))
    return (kpi("month_of_year_overall").select("month", "flights", metric).crossJoin(glob_)
            .withColumn("seasonality_index", F.col(metric) / F.col("overall_value")).orderBy("month"))

def month_matrix(metric="otp_scheduled", dim="overall"):
    """Pivot: hàng = tháng, cột = năm (so cùng tháng qua các năm)."""
    k = kpi(f"month_{dim}")
    years = sorted(r[0] for r in k.select("year").distinct().collect())
    return k.groupBy("month" if dim == "overall" else ["month", dim]).pivot("year", years).agg(F.first(metric)).orderBy("month")

# ── Mức NGÀY ──
def daily_anomalies(metric="delay15_rate", window_days=28, z_thr=3.0, top=15):
    """z-score của từng ngày so với nền 28 ngày trước đó (window theo khoảng ngày); gắn nhãn ngày lễ gần nhất (broadcast join)."""
    d = (kpi("day_overall").withColumn("d", F.make_date("year", "month", "day"))
         .withColumn("dn", F.datediff("d", F.lit("1970-01-01").cast("date"))))
    w = Window.orderBy("dn").rangeBetween(-window_days, -1)
    hol = holiday_df().select(F.col("fl_date").alias("d"), "holiday", "holiday_offset")
    z = (d.withColumn("base_mean", F.avg(metric).over(w)).withColumn("base_std", F.stddev(metric).over(w))
          .withColumn("z", (F.col(metric) - F.col("base_mean")) / F.col("base_std"))
          .join(F.broadcast(hol), "d", "left")
          .withColumn("is_anomaly", F.abs("z") >= z_thr)
          .withColumn("dow", ((F.dayofweek("d") + 5) % 7) + 1))
    return (z.orderBy(F.desc(F.abs("z"))).limit(top)
             .select("d", "dow", "flights", metric, "base_mean", "z", "is_anomaly", "cancel_rate", "avg_arr_delay_pos", "holiday", "holiday_offset"))

def day_in_month(y, m, metric="delay15_rate"):
    """Hướng A: các ngày của MỘT tháng, kèm xếp hạng ngày xấu nhất."""
    k = kpi("day_overall").where((F.col("year") == y) & (F.col("month") == m))
    return (k.withColumn("dow", ((F.dayofweek(F.make_date("year", "month", "day")) + 5) % 7) + 1)
             .withColumn("rank_worst", F.dense_rank().over(Window.orderBy(F.desc(metric))))
             .orderBy("day").select("day", "dow", "flights", "otp_scheduled", "cancel_rate", "delay15_rate", "avg_arr_delay_pos", "rank_worst"))

def calendar_day_across_years(top=15, metric="delay15_rate"):
    """Hướng B-ii: cùng một ngày lịch (ví dụ 25/12) gộp qua các năm."""
    return kpi("calendar_day_overall").orderBy(F.desc(metric)).limit(top).select("month", "day", "n_base_rows", "flights", metric, "cancel_rate", "otp_scheduled")

# ── Mức NĂM / HÃNG ──
def carrier_ranking(year=None):
    k = kpi("year_carrier").where(F.col("flights") >= MIN_FLIGHTS).join(F.broadcast(carrier_lookup()), "carrier", "left")
    if year: k = k.where(F.col("year") == year)
    w = Window.partitionBy("year").orderBy(F.desc("reliability_score"))
    return (k.withColumn("rank", F.dense_rank().over(w)).orderBy("year", "rank")
             .select("year", "rank", "carrier", "carrier_name", "flights", "otp_scheduled", "cancel_rate", "delay15_rate",
                     "avg_delay_when_late", "severe60_rate", "avg_air_gain", "reliability_score"))

def carrier_yoy():
    """Hãng nào cải thiện/xấu đi so với năm trước (self-join theo year-1)."""
    k = kpi("year_carrier").where(F.col("flights") >= MIN_FLIGHTS)
    prev = k.select((F.col("year") + 1).alias("year"), "carrier", F.col("otp_scheduled").alias("otp_prev"), F.col("cancel_rate").alias("cancel_prev"))
    return (k.join(prev, ["year", "carrier"]).withColumn("otp_change", F.col("otp_scheduled") - F.col("otp_prev"))
             .withColumn("cancel_change", F.col("cancel_rate") - F.col("cancel_prev")).orderBy("year", F.desc("otp_change"))
             .select("year", "carrier", "flights", "otp_prev", "otp_scheduled", "otp_change", "cancel_prev", "cancel_rate", "cancel_change"))

def carrier_matrix(metric="otp_scheduled", by="month_of_year"):
    """Pivot: hàng = hãng, cột = tháng-trong-năm hoặc năm."""
    k = kpi(f"{by}_carrier")
    col = "month" if by == "month_of_year" else "year"
    vals = sorted(r[0] for r in k.select(col).distinct().collect())
    return k.where(F.col("flights") >= MIN_FLIGHTS).groupBy("carrier").pivot(col, vals).agg(F.first(metric)).orderBy("carrier")

def tier_distribution(by="carrier"):
    """Phân bố mức độ nghiêm trọng (tầng) theo hãng: pivot tầng → cột, tỷ lệ theo hàng."""
    t = base_tbl("tier_carrier").groupBy(by, "tier").agg(F.sum("count").alias("n"))
    tot = t.groupBy(by).agg(F.sum("n").alias("total"))
    tiers = sorted(r[0] for r in t.select("tier").distinct().collect())
    share = t.join(tot, by).withColumn("share", F.col("n") / F.col("total"))
    return share.groupBy(by, "total").pivot("tier", tiers).agg(F.first("share")).orderBy(by)

# ── Nguyên nhân trễ: unpivot từ dạng rộng sang dạng dài ──
def unpivot_causes(df, keys):
    stack = "stack(5, 'carrier', cause_carrier, 'weather', cause_weather, 'nas', cause_nas, 'security', cause_security, 'late_aircraft', cause_late_aircraft) as (cause, minutes)"
    long = df.select(*keys, "cause_total", F.expr(stack))
    return long.withColumn("share", F.when(F.col("cause_total") > 0, F.col("minutes") / F.col("cause_total")))

# ── Hồ sơ theo giờ / thứ / cự ly / ngày lễ ──
def hour_profile():
    return kpi("hour_overall").orderBy("dep_hour").select("dep_hour", "flights", "otp_scheduled", "delay15_rate", "avg_arr_delay_pos",
                                                         "avg_dep_delay_pos", "cause_late_aircraft_share", "avg_taxi_out", "cancel_rate")
def dow_profile():
    return kpi("dow_overall").orderBy("dow").select("dow", "flights", "otp_scheduled", "delay15_rate", "cancel_rate", "avg_arr_delay_pos")
def distance_profile():
    return kpi("dist_overall").select("dist_bucket", "flights", "otp_scheduled", "avg_air_gain", "avg_dep_delay_pos", "avg_arr_delay_pos")
def holiday_effect():
    """Mức nâng (lift) so với ngày thường theo số ngày cách lễ."""
    h = kpi("holiday_overall")
    b = h.where(F.col("holiday") == "none").select(F.col("delay15_rate").alias("b_delay"), F.col("cancel_rate").alias("b_cancel"))
    return (h.where(F.col("holiday") != "none").crossJoin(b)
             .withColumn("delay15_lift", F.col("delay15_rate") / F.col("b_delay")).withColumn("cancel_lift", F.col("cancel_rate") / F.col("b_cancel"))
             .orderBy("holiday", "holiday_offset").select("holiday", "holiday_offset", "flights", "delay15_rate", "delay15_lift", "cancel_rate", "cancel_lift"))

# ── Sân bay, mạng lưới ──
def airport_tiers(n_large=30, n_medium=70):
    o = kpi("year_origin").groupBy("origin").agg(F.sum("flights").alias("flights"))
    rk = F.row_number().over(Window.orderBy(F.desc("flights")))
    return o.withColumn("rk", rk).withColumn("airport_tier", F.when(F.col("rk") <= n_large, "large").when(F.col("rk") <= n_large + n_medium, "medium").otherwise("small")).select("origin", "rk", "airport_tier")

def tier_kpis():
    """So sánh nhóm sân bay lớn/vừa/nhỏ (broadcast join bảng tier, rollup lại từ KPI cấp sân bay-năm)."""
    j = kpi("year_origin").join(F.broadcast(airport_tiers()), "origin")
    return finalize(rollup(j, ["year", "airport_tier"])).orderBy("year", "airport_tier").select(
        "year", "airport_tier", "flights", "otp_scheduled", "cancel_rate", "avg_dep_delay_pos", "avg_taxi_out", "reliability_score")

def worst_airports(year=None, top=15):
    k = kpi("year_origin").where(F.col("flights") >= MIN_FLIGHTS)
    if year: k = k.where(F.col("year") == year)
    return k.orderBy(F.desc("avg_taxi_out")).limit(top).select("year", "origin", "flights", "otp_scheduled", "cancel_rate", "avg_taxi_out", "avg_dep_delay_pos", "cause_late_aircraft_share")

def concentration():
    """HHI thị phần hãng tại mỗi sân bay xuất phát (>0.25 = tập trung cao, theo ngưỡng DOJ 2.500/10.000)."""
    co = kpi("year_carrier_origin").select("year", "origin", "carrier", "flights")
    tot = co.groupBy("year", "origin").agg(F.sum("flights").alias("airport_flights"))
    j = co.join(tot, ["year", "origin"]).withColumn("share", F.col("flights") / F.col("airport_flights"))
    top = (j.withColumn("rk", F.row_number().over(Window.partitionBy("year", "origin").orderBy(F.desc("share")))).where("rk = 1")
             .select("year", "origin", F.col("carrier").alias("dominant_carrier"), F.col("share").alias("dominant_share")))
    hhi = j.groupBy("year", "origin", "airport_flights").agg(F.sum(F.col("share") ** 2).alias("hhi"))
    return (hhi.join(top, ["year", "origin"])
               .withColumn("hhi_class", F.when(F.col("hhi") > 0.25, "concentrated").when(F.col("hhi") > 0.15, "moderate").otherwise("competitive"))
               .where(F.col("airport_flights") >= MIN_FLIGHTS))

def _pagerank(pdf, d=0.85, iters=60):
    import numpy as np, pandas as pd
    nodes = sorted(set(pdf["origin"]) | set(pdf["dest"])); idx = {n: i for i, n in enumerate(nodes)}; n = len(nodes)
    src = pdf["origin"].map(idx).values; dst = pdf["dest"].map(idx).values; w = pdf["flights"].values.astype(float)
    out = np.zeros(n); np.add.at(out, src, w)
    pr = np.full(n, 1.0 / n)
    for _ in range(iters):
        contrib = np.zeros(n); np.add.at(contrib, dst, pr[src] * w / out[src])
        pr = (1 - d) / n + d * (contrib + pr[out == 0].sum() / n)
    return pd.DataFrame({"airport": nodes, "pagerank": pr})

def airport_network(year=None, top=20):
    """Đồ thị sân bay: cạnh = tuyến có hướng, trọng số = số chuyến. Bậc vào/ra, lưu lượng, mất cân bằng, PageRank (tính ở driver vì đồ thị nhỏ)."""
    e = kpi("year_route")
    if year: e = e.where(F.col("year") == year)
    e = e.groupBy("origin", "dest").agg(F.sum("flights").alias("flights"))
    o = e.groupBy(F.col("origin").alias("airport")).agg(F.count("dest").alias("out_degree"), F.sum("flights").alias("flights_out"))
    i = e.groupBy(F.col("dest").alias("airport")).agg(F.count("origin").alias("in_degree"), F.sum("flights").alias("flights_in"))
    nodes = o.join(i, "airport", "outer").na.fill(0).withColumn("flow_balance", F.col("flights_out") - F.col("flights_in"))
    pr = spark.createDataFrame(_pagerank(e.select("origin", "dest", "flights").toPandas()))
    return nodes.join(pr, "airport", "left").orderBy(F.desc("pagerank")).limit(top)

def route_excess_delay(year=None, hint="merge", top=15):
    """Tuyến nào trễ NHIỀU HƠN mức nền của sân bay xuất phát? Join hai bảng lớn bằng sort-merge join (hint='merge') hoặc broadcast."""
    r = kpi("year_route").where(F.col("flights") >= MIN_FLIGHTS).select("year", "origin", "dest", "flights", F.col("avg_arr_delay_net").alias("route_delay"), F.col("otp_scheduled").alias("route_otp"))
    o = kpi("year_origin").select("year", "origin", F.col("avg_arr_delay_net").alias("origin_baseline"))
    if year: r, o = r.where(F.col("year") == year), o.where(F.col("year") == year)
    j = r.hint(hint).join(o.hint(hint), ["year", "origin"])
    return j.withColumn("excess_delay", F.col("route_delay") - F.col("origin_baseline")).orderBy(F.desc("excess_delay"))

def enrich_routes(year=None):
    """Chuỗi nhiều join: tuyến ⨝ tier sân bay đi ⨝ tier sân bay đến ⨝ nền sân bay đi ⨝ nền sân bay đến, có kiểm tra số dòng (không nhân bản)."""
    r = kpi("year_route")
    if year: r = r.where(F.col("year") == year)
    n0 = r.count()
    t = airport_tiers()
    ob = kpi("year_origin").select("year", F.col("origin").alias("o"), F.col("avg_arr_delay_net").alias("origin_delay"))
    db = kpi("year_origin").select("year", F.col("origin").alias("d"), F.col("avg_arr_delay_net").alias("dest_delay_as_origin"))
    j = (r.join(F.broadcast(t.select("origin", F.col("airport_tier").alias("origin_tier"))), "origin")
          .join(F.broadcast(t.select(F.col("origin").alias("dest"), F.col("airport_tier").alias("dest_tier"))), "dest")
          .join(ob.withColumnRenamed("o", "origin"), ["year", "origin"], "left")
          .join(db.withColumnRenamed("d", "dest"), ["year", "dest"], "left"))
    n1 = j.count()
    print(f"enrich_routes: {n0} dòng trước join, {n1} dòng sau join → {'OK, không nhân bản' if n0 == n1 else 'CẢNH BÁO: số dòng thay đổi!'}")
    return j.withColumn("tier_pair", F.concat_ws("→", "origin_tier", "dest_tier")).select(
        "year", "origin", "dest", "tier_pair", "flights", "otp_scheduled", "avg_arr_delay_net", "origin_delay", "dest_delay_as_origin")

def delay_propagation(df, min_n=100):
    """Lan truyền trễ theo máy bay (cần cột tail): chuyến trước đến trễ có làm chuyến sau khởi hành trễ không? Chỉ xét chuyến liền kề (đến X rồi đi từ X)."""
    if "tail" not in df.columns or df.where(F.col("tail").isNotNull()).limit(1).count() == 0:
        print("ℹ Không có cột số hiệu máy bay (TAIL_NUM) → bỏ qua phân tích lan truyền trễ"); return None
    ts = F.to_timestamp(F.concat_ws(" ", F.col("fl_date").cast("string"), F.format_string("%02d:%02d", "dep_hour", "dep_min")))
    d = df.where(F.col("tail").isNotNull() & (F.col("cancelled") == 0)).withColumn("sched_ts", ts)
    w = Window.partitionBy("tail").orderBy("sched_ts")
    d = (d.withColumn("prev_arr_delay", F.lag("arr_delay_ok").over(w)).withColumn("prev_dest", F.lag("dest").over(w))
          .where((F.col("prev_dest") == F.col("origin")) & F.col("prev_arr_delay").isNotNull() & F.col("dep_delay_ok").isNotNull()))
    d = d.withColumn("prev_state", F.when(F.col("prev_arr_delay") < 0, "1_prev_early").when(F.col("prev_arr_delay") < 15, "2_prev_ontime")
                                   .when(F.col("prev_arr_delay") < 45, "3_prev_late15_44").otherwise("4_prev_late45plus"))
    return (d.groupBy("prev_state").agg(F.count(F.lit(1)).alias("n"), F.avg("dep_delay_ok").alias("avg_dep_delay"),
                                         F.avg((F.col("dep_delay_ok") >= 15).cast("double")).alias("dep_late15_rate"))
             .where(F.col("n") >= min_n).orderBy("prev_state"))

def robust_delay(df, keys, use_trimmed=True):
    """Custom aggregation: trung bình cắt (pandas UDF) so với mean và median. Ghép bằng join vì pandas-UDF không trộn được với hàm gộp thường."""
    base = df.groupBy(*keys).agg(F.count("arr_delay_ok").alias("n"), F.avg("arr_delay_ok").alias("mean"),
                                 F.expr("percentile_approx(arr_delay_ok, 0.5, 10000)").alias("median"))
    if trimmed_mean is None or not use_trimmed:
        return base
    tm = df.groupBy(*keys).agg(trimmed_mean("arr_delay_ok").alias("trimmed_mean"))
    return base.join(tm, keys).withColumn("mean_minus_median", F.col("mean") - F.col("median"))

def check_udf(df):
    """UDF Python vs biểu thức Spark tham chiếu: số dòng không khớp phải bằng 0."""
    t = df.select("tier", severity_udf("arr_delay", "cancelled", "diverted").alias("tier_udf"))
    return t.where(F.col("tier") != F.col("tier_udf")).count(), t.count()


### 9. Bằng chứng cho các kỹ thuật Spark

In [20]:
# ═══════════ BẰNG CHỨNG CHO CÁC KỸ THUẬT SPARK (execution plan, thời gian trước/sau) ═══════════
EVIDENCE = {}

def plan_text(df, mode="formatted"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

@contextlib.contextmanager
def conf(settings):
    old = {k: spark.conf.get(k, None) for k in settings}
    for k, v in settings.items(): spark.conf.set(k, v)
    try: yield
    finally:
        for k, v in old.items():
            spark.conf.unset(k) if v is None else spark.conf.set(k, v)

def save_evidence(name, text):
    os.makedirs(EVIDENCE_DIR, exist_ok=True)
    with open(os.path.join(EVIDENCE_DIR, name + ".txt"), "w", encoding="utf-8") as f:
        f.write(text)

def grab(plan, pattern):
    return re.findall(pattern, plan)

def evidence_pruning():
    """Partition pruning: lọc theo year/month → số phân vùng đọc (tác vụ quét) giảm, plan có PartitionFilters."""
    full = spark.read.parquet(PROCESSED).select("year")
    one = spark.read.parquet(PROCESSED).where((F.col("year") == MONTHS[0][0]) & (F.col("month") == MONTHS[0][1])).select("year")
    res = {"scan_partitions_no_filter": full.rdd.getNumPartitions(), "scan_partitions_with_filter": one.rdd.getNumPartitions()}
    plan = plan_text(one.select(F.count(F.lit(1))))
    res["partition_filters"] = grab(plan, r"PartitionFilters: \[.*?\]")
    save_evidence("pruning", json.dumps(res, indent=2, ensure_ascii=False) + "\n\n" + plan)
    EVIDENCE["pruning"] = res
    return res

def evidence_joins(year=None):
    """Broadcast join vs sort-merge join trên cùng một phép ghép (kết quả phải giống nhau)."""
    out = {}
    with conf({"spark.sql.adaptive.enabled": "false"}):          # tắt AQE để plan hiển thị đúng chiến lược đã chọn
        for hint in ("broadcast", "merge"):
            df = route_excess_delay(year, hint=hint)
            plan = plan_text(df)
            cnt, sec = timed(lambda: df.count())
            out[hint] = {"rows": cnt, "seconds": sec,
                         "strategy": "BroadcastHashJoin" if "BroadcastHashJoin" in plan else ("SortMergeJoin" if "SortMergeJoin" in plan else "other")}
            save_evidence(f"join_{hint}", plan)
    out["same_rowcount"] = out["broadcast"]["rows"] == out["merge"]["rows"]
    EVIDENCE["joins"] = out
    return out

def evidence_cache(df):
    """Cache: chạy 3 truy vấn lặp lại khi không cache vs có cache."""
    def run(d):
        d.groupBy("carrier").agg(F.avg("arr_delay_ok")).collect()
        d.groupBy("origin").agg(F.sum("operated")).collect()
        d.groupBy("dep_hour").agg(F.count(F.lit(1))).collect()
    _, t_no = timed(lambda: [run(df) for _ in range(3)])
    dfc = df.cache()
    _, t_fill = timed(lambda: dfc.count())
    _, t_c = timed(lambda: [run(dfc) for _ in range(3)])
    dfc.unpersist()
    res = {"no_cache_s": t_no, "cache_fill_s": t_fill, "cached_3_runs_s": t_c, "speedup_x": round(t_no / t_c, 2) if t_c else None}
    EVIDENCE["cache"] = res
    return res

def evidence_bucketing(df, buckets=8, table="aero_bts_bucketed", cleanup=True):
    """Bucketing theo hãng: gom nhóm theo cột bucket không cần shuffle (Exchange). Cần catalog (saveAsTable) nên chỉ sống trong session này."""
    cols = ["carrier", "origin", "arr_delay_ok", "operated"]
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    df.select(*cols).write.mode("overwrite").bucketBy(buckets, "carrier").sortBy("carrier").saveAsTable(table)
    q = lambda d: d.groupBy("carrier").agg(F.avg("arr_delay_ok").alias("avg_delay"), F.sum("operated").alias("operated"))
    plain, buck = q(df.select(*cols)), q(spark.table(table))
    with conf({"spark.sql.adaptive.enabled": "false"}):
        p_plain, p_buck = plan_text(plain), plan_text(buck)
    _, t_plain = timed(lambda: plain.collect()); _, t_buck = timed(lambda: buck.collect())
    res = {"exchange_plain": p_plain.count("Exchange"), "exchange_bucketed": p_buck.count("Exchange"),
           "plain_s": t_plain, "bucketed_s": t_buck}
    save_evidence("bucketing_plain", p_plain); save_evidence("bucketing_bucketed", p_buck)
    if cleanup: spark.sql(f"DROP TABLE IF EXISTS {table}")
    EVIDENCE["bucketing"] = res
    return res

def evidence_optimization(months=None):
    """Tối ưu truy vấn: bộ lọc bằng biểu thức Spark (đẩy xuống Parquet, PushedFilters) vs bằng Python UDF (không đẩy xuống được)."""
    raw = load_months(months or MONTHS[:1], verbose=False)
    a = COLMAP.get("arr_delay") or "ARR_DELAY"
    native = raw.where(F.col(a) >= 15).select(F.count(F.lit(1)).alias("late"))
    is_late = F.udf(lambda x: x is not None and x >= 15, T.BooleanType())
    slow = raw.where(is_late(F.col(a))).select(F.count(F.lit(1)).alias("late"))
    res = {}
    for name, d in (("native_filter", native), ("udf_filter", slow)):
        plan = plan_text(d)
        (rows, sec) = timed(lambda: d.collect()[0][0])
        res[name] = {"late_flights": rows, "seconds": sec, "pushed_filters": grab(plan, r"PushedFilters: \[.*?\]"),
                     "read_schema": grab(plan, r"ReadSchema: .*")[:1]}
        save_evidence(f"optimization_{name}", plan)
    res["same_result"] = res["native_filter"]["late_flights"] == res["udf_filter"]["late_flights"]
    EVIDENCE["optimization"] = res
    return res

def evidence_all(df_demo):
    for fn, args in ((evidence_pruning, ()), (evidence_joins, ()), (evidence_cache, (df_demo,)),
                     (evidence_bucketing, (df_demo,)), (evidence_optimization, ())):
        try:
            print(fn.__name__, "→", json.dumps(fn(*args), ensure_ascii=False, default=str))
        except Exception as e:
            print(fn.__name__, "✗", type(e).__name__, str(e)[:300])
    save_evidence("summary", json.dumps(EVIDENCE, indent=2, ensure_ascii=False, default=str))


### 10. Chạy pipeline

In [ ]:
# ═══════════ CHẠY PIPELINE ═══════════
# Bước 1: mỗi tháng → bảng cơ sở + box-plot ngày/tháng.  Bước 2: cuộn lên mọi mức.  Bước 3: box-plot năm/mùa vụ.
RUN_BUILD = True            # False nếu đã build, chỉ muốn đọc kết quả
if RUN_BUILD:
    t0 = time.time()
    print("Bước 1/3 – từng tháng:")
    build_months(MONTHS)
    print("Bước 2/3 – cuộn lên các mức KPI…");  build_kpis()
    print("Bước 3/3 – box-plot năm / mùa vụ:");  build_box_coarse(MONTHS)
    print(f"Hoàn tất sau {round(time.time() - t0, 1)}s → {ANALYTICS}")


Bước 1/3 – từng tháng:


26/10/28 08:08:21 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


  2022-01:    537,902 dòng |   82.3s


26/10/28 08:09:43 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


  2022-02:    495,713 dòng |   67.0s


26/10/28 08:10:50 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


  2022-03:    564,853 dòng |   74.9s


26/10/28 08:12:05 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.
ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.8/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

26/10/28 08:20:17 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from 44152fbd0c4f:46267 in 10000 milliseconds


### 11. Xem kết quả

In [11]:
# ═══════════ XEM KẾT QUẢ (mỗi mục là một câu hỏi phân tích) ═══════════
def section(t): print("\n" + "═" * 8 + " " + t + " " + "═" * 8)

section("0. Tổng quan theo năm (n_base_rows = số ngày có dữ liệu; năm cuối có thể chưa đủ)"); peek(overview())

section("MỨC THÁNG – chuỗi tháng, trung bình trượt 3 tháng, so cùng kỳ năm trước"); peek(monthly_trend("otp_scheduled"), 60)
section("MỨC THÁNG – chỉ số mùa vụ (tháng-trong-năm so với toàn kỳ)"); peek(seasonality("delay15_rate"), 12)
section("MỨC THÁNG – ma trận tháng × năm (pivot)"); peek(month_matrix("otp_scheduled"), 12)

section("MỨC NGÀY – hướng A: các ngày trong một tháng"); peek(day_in_month(*MONTHS[0]), 31)
section("MỨC NGÀY – hướng B: ngày bất thường (z-score so với nền 28 ngày) và ngày lịch xấu nhất qua các năm")
peek(daily_anomalies()); peek(calendar_day_across_years())

section("MỨC NĂM / HÃNG – xếp hạng độ tin cậy, thay đổi so với năm trước"); peek(carrier_ranking(), 40); peek(carrier_yoy(), 40)
section("HÃNG – ma trận hãng × tháng-trong-năm (pivot)"); peek(carrier_matrix("otp_scheduled", "month_of_year"), 40)
section("HÃNG – phân bố mức độ nghiêm trọng (tầng 0–8)"); peek(tier_distribution(), 40)

section("NGUYÊN NHÂN TRỄ – dạng dài (unpivot) theo năm"); peek(unpivot_causes(kpi("year_overall"), ["year"]), 20)

section("HỒ SƠ – theo giờ khởi hành (trễ tích lũy trong ngày)"); peek(hour_profile(), 24)
section("HỒ SƠ – thứ trong tuần / cự ly bay / ảnh hưởng ngày lễ"); peek(dow_profile(), 7); peek(distance_profile()); peek(holiday_effect(), 50)

section("SÂN BAY – nhóm lớn/vừa/nhỏ, sân bay tắc nghẽn mặt đất, mức độ tập trung hãng (HHI)")
peek(tier_kpis(), 20); peek(worst_airports()); peek(concentration().orderBy(F.desc("hhi")), 20)
section("MẠNG LƯỚI – bậc vào/ra, lưu lượng, mất cân bằng, PageRank"); peek(airport_network(), 20)
section("TUYẾN – trễ vượt mức nền sân bay đi (sort-merge join) và bảng tuyến đã làm giàu (nhiều join)")
peek(route_excess_delay(), 15); peek(enrich_routes(), 15)

section("BOX-PLOT – thống kê phân phối delay (năm × toàn bộ)"); peek(box("year_overall_delay").orderBy("year", "metric"), 40)
section("BOX-PLOT – delay đến của chuyến trễ, theo hãng (chỉ nhóm đủ lớn)")
peek(box("year_carrier_delay").where((F.col("metric") == "arr_delay_late") & F.col("reliable"))
     .select("year", "carrier", "n", "min", "q1", "median", "q3", "whisker_high", "p95", "p99", "max", "out_high_rate").orderBy("year", F.desc("median")), 40)



════════ 0. Tổng quan theo năm (n_base_rows = số ngày có dữ liệu; năm cuối có thể chưa đủ) ════════


,year,n_base_rows,flights,flights_per_day,cancel_rate,divert_rate,otp_operated,otp_scheduled,avg_arr_delay_net,avg_arr_delay_pos,avg_delay_when_late,delay15_rate,severe60_rate,severe180_rate,early_rate,avg_air_gain,reliability_score
0,2022,90,1598468,17760.755556,0.039872,0.002095,0.797907,0.764421,5.320524,14.52282,66.969407,0.202093,0.067377,0.011891,0.627653,6.25994,87.540544



════════ MỨC THÁNG – chuỗi tháng, trung bình trượt 3 tháng, so cùng kỳ năm trước ════════


26/10/28 07:24:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 07:24:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 07:24:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 07:24:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 07:24:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 07:24:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/28 0

,year,month,flights,otp_scheduled,otp_scheduled_prev_year,otp_scheduled_ma3,yoy_change
0,2022,1,537902,0.755671,NaN,0.755671,NaN
1,2022,2,495713,0.767646,NaN,0.761658,NaN
2,2022,3,564853,0.769924,NaN,0.764414,NaN


...

,year,origin,dest,tier_pair,flights,otp_scheduled,avg_arr_delay_net,origin_delay,dest_delay_as_origin
0,2022,KTN,SEA,small→large,269,0.791822,1.299213,3.860279,1.283701
1,2022,BOS,SLC,large→large,229,0.733624,14.444954,7.556351,1.090264
2,2022,DEN,IAD,large→medium,701,0.656205,14.369822,11.684208,6.849206
3,2022,LGA,STL,large→medium,1079,0.721965,0.527291,5.939735,3.702154
4,2022,STL,DAL,medium→large,388,0.693299,2.464789,3.702154,6.236986
5,2022,MSY,STL,medium→medium,164,0.829268,2.483871,5.169406,3.702154
6,2022,BRD,MSP,small→large,150,0.806667,35.835616,35.835616,4.283093
7,2022,FAY,CLT,small→large,178,0.786517,2.916168,7.670886,1.153182
8,2022,CLT,DSM,large→medium,237,0.839662,-3.302222,1.153182,6.797329
9,2022,FLL,BUF,large→medium,172,0.726744,11.688623,9.246611,12.364960



════════ BOX-PLOT – thống kê phân phối delay (năm × toàn bộ) ════════


,year,n,min,max,mean,std,p05,q1,median,q3,...,fence_low,fence_high,whisker_low,whisker_high,n_out_low,n_out_high,out_low_rate,out_high_rate,metric,reliable
0,2022,1531385,-100.0,2934.0,5.320524,53.020429,-28.0,-16.0,-6.0,9.0,...,-53.5,46.5,-53.0,46.0,1106,133614,0.000722,0.087250,arr_delay_all,True
1,2022,309482,15.0,2934.0,66.969407,92.704704,16.0,24.0,40.0,75.0,...,-52.5,151.5,15.0,151.0,0,25612,0.000000,0.082758,arr_delay_late,True
2,2022,1534734,-78.0,2966.0,11.623765,50.664785,-10.0,-5.0,-2.0,9.0,...,-26.0,30.0,-26.0,30.0,238,196046,0.000155,0.127739,dep_delay_all,True
3,2022,312332,15.0,2966.0,67.132827,92.807741,16.0,24.0,40.0,76.0,...,-54.0,154.0,15.0,154.0,0,24961,0.000000,0.079918,dep_delay_late,True



════════ BOX-PLOT – delay đến của chuyến trễ, theo hãng (chỉ nhóm đủ lớn) ════════


,year,carrier,n,min,q1,median,q3,whisker_high,p95,p99,max,out_high_rate
0,2022,B6,20033,15.0,29.0,53.0,102.0,211.0,234.0,399.0,2029.0,0.064494
1,2022,YV,5602,15.0,26.0,48.0,96.0,201.0,263.0,840.0,1818.0,0.077651
2,2022,F9,11027,15.0,26.0,46.0,86.0,176.0,207.0,370.0,1284.0,0.070373
3,2022,OH,12266,15.0,26.0,46.0,92.0,191.0,228.0,436.0,1444.0,0.072477
4,2022,OO,31786,15.0,25.0,44.0,90.0,187.0,254.0,712.0,2166.0,0.085635
5,2022,9E,9625,15.0,25.0,43.0,86.0,177.0,239.0,686.0,1368.0,0.081558
6,2022,YX,16128,15.0,25.0,43.0,80.0,162.0,189.0,345.0,1306.0,0.070064
7,2022,NK,13761,15.0,25.0,43.0,79.0,160.0,182.0,332.0,1294.0,0.068672
8,2022,G4,9157,15.0,25.0,41.0,74.0,147.0,184.0,352.0,1364.0,0.083543
9,2022,UA,26987,15.0,24.0,40.0,75.0,151.0,190.0,385.0,1451.0,0.079038


### 12. Phần dùng dữ liệu thô của tập nhỏ và bằng chứng kỹ thuật

In [12]:
# ═══════════ PHẦN DÙNG DỮ LIỆU THÔ CỦA MỘT TẬP NHỎ: lan truyền trễ, custom aggregation, UDF, bằng chứng kỹ thuật ═══════════
DEMO_MONTHS = MONTHS[:1]                       # 1 tháng là đủ để minh họa; tăng lên nếu máy đủ RAM
df_demo = prepare(load_months(DEMO_MONTHS))

section("Lan truyền trễ theo máy bay (chuyến trước đến trễ → chuyến sau đi trễ?)"); peek(delay_propagation(df_demo))
section("UDF so với biểu thức tham chiếu (số dòng KHÔNG khớp / tổng)"); print(check_udf(df_demo))
section("Custom aggregation: trimmed mean vs mean vs median")
try:
    peek(robust_delay(df_demo, ["year", "carrier"]).orderBy("year", "carrier"), 20)
except Exception as e:
    print("⚠ Không chạy được pandas UDF ở đây:", type(e).__name__, "(cần pandas + pyarrow và Java 8/11/17). Hiển thị mean/median thay thế:")
    peek(robust_delay(df_demo, ["year", "carrier"], use_trimmed=False).orderBy("year", "carrier"), 20)

section("BẰNG CHỨNG KỸ THUẬT (partition pruning, broadcast/sort-merge join, cache, bucketing, tối ưu truy vấn)")
evidence_all(df_demo)
print("\nPlan chi tiết đã lưu trong thư mục:", os.path.abspath(EVIDENCE_DIR))

section("Các bảng kết quả đã ghi")
for sub in ("base", "kpi", "box"):
    path, fs = _hpath(f"{ANALYTICS}/{sub}")
    for st in fs.listStatus(path):
        print(f"  {sub}/{st.getPath().getName()}")


26/10/28 07:36:41 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


ℹ Cột tùy chọn không có (chỉ số liên quan sẽ là null): ['taxi_out', 'taxi_in', 'crs_elapsed', 'actual_elapsed']

════════ Lan truyền trễ theo máy bay (chuyến trước đến trễ → chuyến sau đi trễ?) ════════


,prev_state,n,avg_dep_delay,dep_late15_rate
0,1_prev_early,320288,1.813187,0.090887
1,2_prev_ontime,75990,7.988209,0.178866
2,3_prev_late15_44,50151,20.771550,0.459891
3,4_prev_late45plus,43407,58.644758,0.633654



════════ UDF so với biểu thức tham chiếu (số dòng KHÔNG khớp / tổng) ════════


(0, 537902)

════════ Custom aggregation: trimmed mean vs mean vs median ════════


...

evidence_pruning → {"scan_partitions_no_filter": 329, "scan_partitions_with_filter": 7, "partition_filters": ["PartitionFilters: [isnotnull(year#383149), isnotnull(month#383151), (year#383149 = 2022), (month#383151 = 1)]"]}
evidence_joins → {"broadcast": {"rows": 2568, "seconds": 0.22, "strategy": "BroadcastHashJoin"}, "merge": {"rows": 2568, "seconds": 0.53, "strategy": "SortMergeJoin"}, "same_rowcount": true}


evidence_cache → {"no_cache_s": 17.97, "cache_fill_s": 5.75, "cached_3_runs_s": 2.19, "speedup_x": 8.21}


evidence_bucketing → {"exchange_plain": 4, "exchange_bucketed": 0, "plain_s": 1.7, "bucketed_s": 0.45}


26/10/28 07:37:46 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `month` already exists. Consider to choose another name or rename the existing column.


evidence_optimization → {"native_filter": {"late_flights": 97052, "seconds": 1.4, "pushed_filters": ["PushedFilters: [IsNotNull(ARR_DELAY), GreaterThanOrEqual(ARR_DELAY,15.0)]"], "read_schema": ["ReadSchema: struct<ARR_DELAY:double>"]}, "udf_filter": {"late_flights": 97052, "seconds": 1.84, "pushed_filters": [], "read_schema": ["ReadSchema: struct<ARR_DELAY:double>"]}, "same_result": true}

Plan chi tiết đã lưu trong thư mục: /opt/spark_jobs/evidence

════════ Các bảng kết quả đã ghi ════════
  base/day_carrier
  base/day_origin
  base/day_overall
  base/month_carrier_origin
  base/month_route
  base/prof_dist
  base/prof_dow
  base/prof_holiday
  base/prof_hour
  base/prof_hour_carrier
  base/tier_carrier
  kpi/calendar_day_carrier
  kpi/calendar_day_origin
  kpi/calendar_day_overall
  kpi/day_carrier
  kpi/day_origin
  kpi/day_overall
  kpi/dist_overall
  kpi/dow_by_year
  kpi/dow_overall
  kpi/holiday_overall
  kpi/hour_by_year
  kpi/hour_carrier
  kpi/hour_overall
  kpi/month_carri